# Explore the restriction-enzyme catalogue

Look up EcoRI, search for related names and inspect entries with uncertain cleavage data. The desktop Digest view uses the same read-only `RestrictionCatalogue` reader. [Catalogue sources and terminology](../../databases/enzymes/README.md)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "strandfolio" / "__init__.py").is_file())
sys.path.insert(0, str(root))

from databases.enzymes.build_catalogue import buildDatabase
from strandfolio.storage.catalogues import RestrictionCatalogue

## Open a catalogue

The local catalogue is optional in a fresh checkout. If it is missing, build a copy from the installed Biopython dictionary in the notebook's output directory. The release metadata identifies the source used for these queries.

In [ ]:
databasePath = root / "databases/enzymes/restriction_enzymes.sqlite3"
if not databasePath.exists():
    runId = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    databasePath = root / "docs/notebooks/_outputs/E06" / runId / databasePath.name
    buildDatabase(databasePath)

with RestrictionCatalogue(databasePath) as catalogue:
    release = catalogue.release
    counts = catalogue.summary()

print(databasePath.relative_to(root))
print(f"Biopython {release.biopythonVersion} · {counts['enzymes']:,} enzymes")
print(release.sourceReleaseDescription)

## Look up an enzyme and search by name

`get()` returns a single entry by name. `search()` matches part of a name or recognition sequence. Here the search returns up to five entries with recorded cleavage positions.

In [ ]:
with RestrictionCatalogue(databasePath) as catalogue:
    ecoRI = catalogue.get("EcoRI")
    matches = catalogue.search("Eco", cleavageKnown=True, limit=5)

print(f"{ecoRI.name} recognizes {ecoRI.recognitionSite} ({ecoRI.cutNotation}).")
for enzyme in matches:
    print(f"{enzyme.name:<10} {enzyme.recognitionSite:<14} {enzyme.overhangType}")

## Inspect an unknown cut and a flagged entry

HpyUM037X has alternative recognition motifs and no recorded cleavage positions. TscAI retains the source's warning about conflicting overhang data. Check an entry's `cleavagePairs` and `issues` before interpreting its geometry.

In [ ]:
with RestrictionCatalogue(databasePath) as catalogue:
    unknown = catalogue.get("HpyUM037X")
    flagged = catalogue.get("TscAI")

print(f"{unknown.name}: {unknown.recognitionSite}, recorded cut pairs = {unknown.cleavagePairs}")
for issue in flagged.issues:
    print(f"{flagged.name}: {issue.detail}")